# Colab runner for the preprocessing search

This notebook runs `experiments/run_preprocessing.py` on a Colab GPU.

Before you start:
1. On your Mac, zip the project folder (it should contain `ocular`, `experiments`, `data`, `outputs`) and upload the single zip to your Google Drive. A zip uploads far faster than the 84k separate image files.
2. In Colab, set the runtime to a GPU (Runtime, Change runtime type, GPU).
3. Run the cells below in order.

Results are written directly to a CSV on your Drive, so if the session drops you can rerun and it continues from where it stopped.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Unzip the project to the local disk

Set `DRIVE_ZIP` to the path of your uploaded zip. The project is extracted to the fast local disk, because reading the training images straight from Drive would be very slow.

In [ ]:
import zipfile
import os

DRIVE_ZIP = "/content/drive/MyDrive/ocular-scan-ai.zip"   # <-- set to your zip
EXTRACT_TO = "/content"

with zipfile.ZipFile(DRIVE_ZIP) as z:
    z.extractall(EXTRACT_TO)

# The project root is the folder that contains ocular and experiments.
PROJECT = "/content/ocular-scan-ai"                        # <-- adjust if your zip nests differently
print("contents:", os.listdir(PROJECT))

## 3. Install the few packages Colab is missing

Torch and torchvision come preinstalled with CUDA, so only a couple of small packages are needed.

In [ ]:
!pip install -q scipy scikit-learn tqdm pillow

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Set Runtime, Change runtime type, GPU, then rerun.")

## 4. Check that the package imports and the device is the GPU

In [ ]:
import sys
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

from ocular import config, data, model, train, eval
print("classes:", config.CLASSES)
print("device:", train.get_device())

## 5. Run the preprocessing search

Every configuration builds its cache, trains a ResNet50, and is scored on OCTDL and the clinic. Results go to a CSV on your Drive. The run is resumable, so a configuration already in the CSV is skipped on a rerun.

Start with a one-configuration pilot to confirm the timing, then run all eleven.

In [ ]:
RESULTS = "/content/drive/MyDrive/preprocessing_results.csv"

# Pilot: one configuration, three epochs, to measure the per-epoch time on this GPU.
!PYTHONPATH={PROJECT} python experiments/run_preprocessing.py --runs 4 --epochs 3 --num-workers 4 --out {RESULTS}

## 6. Run all eleven configurations

Once the pilot timing looks fine, run the whole phase. Rerunning is safe, finished configurations are skipped.

In [ ]:
!PYTHONPATH={PROJECT} python experiments/run_preprocessing.py --epochs 8 --num-workers 4 --out {RESULTS}

## 7. Look at the results

In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS)
df.sort_values("octdl_recall_DRUSEN", ascending=False)